# 1. Generate and validate synthetic PON data

Run this first. No external dataset or Google Drive is required. Settings live in `configs/synthetic.yml`. Existing output is verified and reused; change the output path for a new experiment.

In [ ]:
from pathlib import Path
import json
import sys
import yaml

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))
EXPERIMENT = yaml.safe_load(
    (ROOT / "configs/synthetic_experiment.yml").read_text()
)
DATASET = ROOT / EXPERIMENT["dataset"]
RUN = ROOT / EXPERIMENT["output"]


In [ ]:
from telco_anomaly.synthetic import generate_dataset, load_config
from telco_anomaly.synthetic_validation import validate_dataset, save_report

config, output = load_config(ROOT / "configs/synthetic.yml")
assert ROOT / output == DATASET, "Align generator and experiment dataset paths"
if not DATASET.exists():
    generate_dataset(config, DATASET)
from dataclasses import asdict

manifest = json.loads((DATASET / "manifest.json").read_text())
assert manifest["config"] == asdict(config), "Existing dataset uses different settings"
report = validate_dataset(DATASET)
display(report["checks"])
assert report["checks"].status.eq("pass").all(), "Resolve failed invariants first"
display(report["summary"])
REPORT = ROOT / "outputs/synthetic_validation_v5"
if not REPORT.exists():
    save_report(report, REPORT)

A structural pass means internally consistent simulation, not proven field realism. Distribution summaries exclude the final 15%. Full-file checksums and structural checks may inspect all rows without exposing final model performance. Read `SYNTHETIC_REVIEW.md` before interpreting scores.

In [ ]:
# Independent draws check the assumed event laws, not field realism.
from telco_anomaly.synthetic_validation import audit_event_sampling

sampling = audit_event_sampling(config, repetitions=100)
display(sampling)
sampling.to_csv(REPORT / "event_sampling.csv", index=False)